In [ ]:
!pip install git+https://github.com/huggingface/transformers@v4.51.3-Qwen2.5-Omni-preview
!pip install qwen-omni-utils
!pip install openai pydub pandas

  Cloning https://github.com/huggingface/transformers (to revision v4.51.3-Qwen2.5-Omni-preview) to /tmp/pip-req-build-p274avjw
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/transformers /tmp/pip-req-build-p274avjw
  Running command git checkout -q f551466201fb4089d6df9dc55d80f0edbd149d85
  Resolved https://github.com/huggingface/transformers to commit f551466201fb4089d6df9dc55d80f0edbd149d85
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 81.5 MB/s eta 0:00:00
  Created wheel for transformers: filename=transformers-4.52.0.dev0-py3-none-any.whl size=11252796 sha256=2f2561ccc12a90f1140e3bda0086067b3da2be7f49c93e4206abecd3d49f1811
  Stored in directory: /tmp/pip-ephem-wheel-cache-tuffr0hm/wheels/f1/49/78/3136007fa90a8de6e205f1013

モデル定義

In [ ]:
import torch
import logging
from transformers import Qwen2_5OmniForConditionalGeneration, Qwen2_5OmniProcessor
from qwen_omni_utils import process_mm_info

logging.getLogger().setLevel(logging.ERROR)

print("Qwenモデルをダウンロード＆ロード中")

model_path = "Qwen/Qwen2.5-Omni-7B"

model = Qwen2_5OmniForConditionalGeneration.from_pretrained(
    model_path,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation="sdpa",
)

processor = Qwen2_5OmniProcessor.from_pretrained(model_path)

print("完了")

🚀 Qwenモデルをダウンロード＆ロードしています...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

Unrecognized keys in `rope_scaling` for 'rope_type'='default': {'mrope_section'}


model.safetensors.index.json: 0.00B [00:00, ?B/s]

...

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/832 [00:00<?, ?B/s]

chat_template.json: 0.00B [00:00, ?B/s]

✅ モデルの準備が完了しました！


一括音声処理 & VADスコア解析（CSV出力）

In [ ]:
import re
import os
import glob
import librosa
import logging
import pandas as pd
from pydub import AudioSegment
from google.colab import drive

logging.getLogger().setLevel(logging.ERROR)

drive.mount('/content/drive')

INPUT_DIR = "/content/drive/MyDrive/VAD_Experiment/inputs"
OUTPUT_DIR = "/content/drive/MyDrive/VAD_Experiment/outputs"

CHUNK_SEC = 10       # セグメントサイズ(s)
OVERLAP_SEC = 5      # オーバーラップ(s)

os.makedirs(OUTPUT_DIR, exist_ok=True)
audio_files = sorted(glob.glob(f"{INPUT_DIR}/*.wav"))

if not audio_files:
    print(f"'{INPUT_DIR}' にWAVファイルが見つかりません。")
else:
    print(f"{len(audio_files)} 個の音声ファイルを見つけました。処理を開始します。")

system_prompt = (
    "You are Qwen, a virtual human developed by the Qwen Team, Alibaba Group, capable of perceiving auditory and visual inputs, as well as generating text and speech. "
    "CRITICAL INSTRUCTION: You are now functioning as a strict VAD (Valence, Arousal, Dominance) scoring API. "
    "You MUST output exactly ONE JSON array containing three float numbers (1.0 to 9.0) like [5.0, 6.0, 5.0]. "
    "The audio WILL contain game sound effects (gunshots, explosions, BGM) and Japanese speech. "
    "Do NOT describe the audio. Do NOT write any conversational text. "
    "If you output ANY words other than the JSON array, the system will critically fail."
)

for audio_path in audio_files:
    filename = os.path.basename(audio_path)
    base_name = os.path.splitext(filename)[0]
    print(f"\n 処理開始: {filename}")

    audio = AudioSegment.from_file(audio_path, format="wav")
    chunk_length_ms = CHUNK_SEC * 1000
    overlap_ms = OVERLAP_SEC * 1000
    step_ms = chunk_length_ms - overlap_ms

    vad_results = {"Time": [], "V": [], "A": [], "D": []}

    total_chunks = len(range(0, len(audio), step_ms))
    chunk_idx = 1

    for i in range(0, len(audio), step_ms):
        chunk = audio[i : i + chunk_length_ms]
        actual_sec = int(i / 1000.0)
        time_str_print = f"{actual_sec//60:02d}:{actual_sec%60:02d}"

        temp_chunk_path = "temp_chunk.wav"
        chunk.export(temp_chunk_path, format="wav")

        audio_array, _ = librosa.load(temp_chunk_path, sr=16000)

        messages = [
            {"role": "system", "content": [{"type": "text", "text": system_prompt}]},
            {"role": "user", "content": [{"type": "text", "text": "Analyze the Valence, Arousal, and Dominance (1.0-9.0) of this audio. Force the output to be ONLY a JSON array."}]},
            {"role": "assistant", "content": [{"type": "text", "text": "[3.5, 8.5, 4.0]"}]},
            {"role": "user", "content": [{"type": "audio", "audio_url": temp_chunk_path}, {"type": "text", "text": "Analyze this audio. Output the array ONLY."}]}
        ]

        text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = processor(text=text, audio=[audio_array], return_tensors="pt", padding=True).to("cuda")

        generate_ids = model.generate(**inputs, max_new_tokens=20, return_audio=False)
        generate_ids = [cur_ids[len(inputs.input_ids[0]):] for cur_ids in generate_ids]
        output_text = processor.batch_decode(generate_ids, skip_special_tokens=True)[0]

        match = re.search(r'\[([0-9.]+),\s*([0-9.]+),\s*([0-9.]+)\]', output_text)

        if match:
            v, a, d = map(float, match.groups())
        else:
            print(f"抽出失敗（{output_text}）")
            v, a, d = 5.0, 5.0, 5.0

        vad_results["Time"].append(actual_sec)
        vad_results["V"].append(v)
        vad_results["A"].append(a)
        vad_results["D"].append(d)

        print(f"[{chunk_idx}/{total_chunks}] {time_str_print} ({actual_sec}s) -> V:{v}, A:{a}, D:{d}")
        chunk_idx += 1

    df = pd.DataFrame(vad_results).T
    csv_path = f"{OUTPUT_DIR}/{base_name}_VAD.csv"
    df.to_csv(csv_path, header=False)
    print(f"保存完了: {csv_path}")

print("\n 解析完了")

/usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):


Mounted at /content/drive
✅ 20 個の音声ファイルを見つけました。処理を開始します。

🎬 処理開始: rec01.wav
  🔄 [1/34] 00:00 (0s) 完了 -> V:6.0, A:7.0, D:5.0
  🔄 [2/34] 00:05 (5s) 完了 -> V:2.0, A:7.0, D:3.0
  🔄 [3/34] 00:10 (10s) 完了 -> V:6.0, A:7.0, D:5.0
  🔄 [4/34] 00:15 (15s) 完了 -> V:7.0, A:8.0, D:6.0
  🔄 [5/34] 00:20 (20s) 完了 -> V:7.0, A:8.0, D:6.0
  🔄 [6/34] 00:25 (25s) 完了 -> V:2.0, A:7.0, D:3.0
  🔄 [7/34] 00:30 (30s) 完了 -> V:6.0, A:8.0, D:7.0
  🔄 [8/34] 00:35 (35s) 完了 -> V:2.0, A:7.0, D:3.0
  🔄 [9/34] 00:40 (40s) 完了 -> V:5.0, A:7.0, D:6.0
  🔄 [10/34] 00:45 (45s) 完了 -> V:3.0, A:7.0, D:5.0
  🔄 [11/34] 00:50 (50s) 完了 -> V:3.0, A:6.0, D:5.0
  🔄 [12/34] 00:55 (55s) 完了 -> V:3.0, A:7.0, D:5.0
  🔄 [13/34] 01:00 (60s) 完了 -> V:2.0, A:5.0, D:3.0
  🔄 [14/34] 01:05 (65s) 完了 -> V:2.0, A:5.0, D:3.0
  🔄 [15/34] 01:10 (70s) 完了 -> V:2.0, A:3.0, D:5.0
  🔄 [16/34] 01:15 (75s) 完了 -> V:2.0, A:7.0, D:3.0
  🔄 [17/34] 01:20 (80s) 完了 -> V:7.0, A:8.0, D:6.0
  🔄 [18/34] 01:25 (85s) 完了 -> V:7.0, A:8.0, D:6.0
  🔄 [19/34] 01:30 (90s) 完了 -> V:7.0